<a href="https://colab.research.google.com/github/ChaMooKwan/SunMoon-Univ.-Machin-Learning-Project/blob/main/%EA%B8%B0%EA%B3%84%ED%95%99%EC%8A%B5%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8_ko_BERT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch sentencepiece

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

# 1. 토크나이저와 기본 KoBERT 모델 로드 (trust_remote_code=True 필수)
model_name = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
bert_base = AutoModel.from_pretrained(model_name)

# 2. 멀티 태스크(ABSA)를 위한 커스텀 모델 클래스 정의
class ABSAKoBERT(nn.Module):
    def __init__(self, bert_model):
        super(ABSAKoBERT, self).__init__()
        self.bert = bert_model

        # BERT의 출력 벡터 크기 (기본 768)
        hidden_size = self.bert.config.hidden_size

        # 첫 번째 머리 (Head): 속성 존재 여부 (0 or 1 -> 2개 클래스)
        self.confidence_classifier = nn.Linear(hidden_size, 2)

        # 두 번째 머리 (Head): 감성 극성 (부정0, 중립1, 긍정2 -> 3개 클래스)
        self.polarity_classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        # 1) 입력값을 BERT에 통과시킴
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # 2) [CLS] 토큰의 벡터만 추출 (인덱스 0번. 문장 전체의 의미를 담고 있음)
        cls_token_output = outputs.last_hidden_state[:, 0, :]

        # 3) 추출한 벡터를 각각의 분류기로 보냄
        confidence_logits = self.confidence_classifier(cls_token_output)
        polarity_logits = self.polarity_classifier(cls_token_output)

        return confidence_logits, polarity_logits

# 3. 모델 객체 생성 및 GPU(또는 CPU) 할당
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ABSAKoBERT(bert_base).to(device)

print("모델 로드 및 구조 설정 완료! 현재 사용 기기:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

모델 로드 및 구조 설정 완료! 현재 사용 기기: cuda


In [ ]:
import torch
from torch.utils.data import Dataset

class ABSADataset(Dataset):
    def __init__(self, texts, aspects, confidences, polarities, tokenizer, max_len=128):
        self.texts = texts
        self.aspects = aspects
        self.confidences = confidences
        self.polarities = polarities
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        aspect = str(self.aspects[item])

        # 1. 텍스트와 속성을 함께 토큰화 (예: [CLS] 문장 [SEP] 속성 [SEP])
        encoding = self.tokenizer(
            text,
            aspect,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        # 2. 감성 라벨 변환 (-1, 0, 1 -> 0, 1, 2)
        # PyTorch의 CrossEntropyLoss는 음수 인덱스를 받을 수 없기 때문입니다.
        # 이 부분은 이미 데이터프레임에서 처리되었으므로 추가적인 +1은 필요 없습니다.
        polarity_label = self.polarities[item]

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            # 커스텀 모델에 전달할 두 개의 정답
            'confidence_labels': torch.tensor(self.confidences[item], dtype=torch.long),
            'polarity_labels': torch.tensor(polarity_label, dtype=torch.long)
        }

### 데이터 세팅

In [ ]:
#데이터 불러오기
import pandas as pd

train_data = pd.read_csv('Training_비율통합.csv')
valid_data = pd.read_csv('Validation_비율통합.csv')

train_data['AspectConfidence'] = 1
valid_data['AspectConfidence'] = 1

In [ ]:
#데이터셋 생성
train_data = train_data[['SentimentText', 'Aspect', 'SentimentPolarity', 'AspectConfidence']]
train_data['SentimentPolarity'] = train_data['SentimentPolarity'] + 1

valid_data = valid_data[['SentimentText', 'Aspect', 'SentimentPolarity', 'AspectConfidence']]
valid_data['SentimentPolarity'] = valid_data['SentimentPolarity'] + 1

def create_negative_aspect_samples(data):
    negative_data = []

    # 각 aspect의 positive 개수
    aspect_counts = train_data['Aspect'].value_counts()

    for target_aspect, count in aspect_counts.items():

        # target_aspect가 아닌 문장들만 후보
        candidate_rows = data[
            data['Aspect'] != target_aspect
        ]

        # 중복 허용 여부
        sampled_rows = candidate_rows.sample(
            n=count,
            replace=len(candidate_rows) < count,
            random_state=42
        )

        for _, row in sampled_rows.iterrows():

            negative_data.append({
                "SentimentText": row["SentimentText"],
                "Aspect": target_aspect,
                "SentimentPolarity": -1,   # mask
                "AspectConfidence": 0
            })

    return negative_data

negative_train_data = create_negative_aspect_samples(train_data)
negative_valid_data = create_negative_aspect_samples(valid_data)

#기존 데이터 셋과 합치기
train_data = pd.concat([train_data, pd.DataFrame(negative_train_data)], ignore_index=True)
valid_data = pd.concat([valid_data, pd.DataFrame(negative_valid_data)], ignore_index=True)

#이제 train_data와 valid_data를 사용하기만 하면 됨.

In [ ]:
texts = train_data['SentimentText'].tolist()
aspects = train_data['Aspect'].tolist()
confidence = train_data['AspectConfidence'].tolist()
polarities = train_data['SentimentPolarity'].tolist()

test_texts = valid_data['SentimentText'].tolist()
test_aspects = valid_data['Aspect'].tolist()
test_confidence = valid_data['AspectConfidence'].tolist()
test_polarities = valid_data['SentimentPolarity'].tolist()

# 데이터셋 생성
test_dataset = ABSADataset(test_texts, test_aspects, test_confidence, test_polarities, tokenizer)
print("테스트 데이터셋 준비 완료! 첫 번째 데이터 샘플:\n", test_dataset[0])
train_dataset = ABSADataset(texts, aspects, confidence, polarities, tokenizer)
print("데이터셋 준비 완료! 첫 번째 데이터 샘플:\n", train_dataset[0])

테스트 데이터셋 준비 완료! 첫 번째 데이터 샘플:
 {'input_ids': tensor([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1, 517, 493,   0, 517, 492,   0, 517,   0, 490, 494,   0, 517,   0,
        493,   0, 490,   0, 517,   0, 491,   0,   3, 517,   0, 491, 494,   0,
          3,   2]), 'attention_mask': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class ABSATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. 입력 데이터 추출
        input_ids = inputs.get("input_ids")
        attention_mask = inputs.get("attention_mask")
        confidence_labels = inputs.get("confidence_labels")
        polarity_labels = inputs.get("polarity_labels")

        # 2. 모델 예측값 뽑기 (우리가 만든 ABSAKoBERT의 forward 결과)
        confidence_logits, polarity_logits = model(input_ids, attention_mask)

        # 3. 각각의 손실(Loss) 계산 함수 선언
        # polarity_labels에 -1이 있을 수 있으므로 ignore_index=-1을 추가
        loss_fct_confidence = nn.CrossEntropyLoss(reduction='none')
        loss_fct_polarity = nn.CrossEntropyLoss(reduction='none', ignore_index=-1)

        # 4. 오차 계산
        loss_confidence = loss_fct_confidence(confidence_logits, confidence_labels).mean()

        # 감성 오차를 구한 뒤
        loss_polarity = loss_fct_polarity(polarity_logits, polarity_labels)

        # 💡 핵심 로직: 정답(confidence_labels)이 1인 경우만 감성 오차를 살리고, 0이면 곱해서 오차를 0으로 날려버림!
        # ignore_index=-1 설정으로 인해 -1 라벨은 이미 계산에서 제외되지만, 이 로직은 confidence_labels=0일 때도 polarity loss를 무시하도록 합니다.
        loss_polarity = (loss_polarity * confidence_labels).mean()

        # 5. 최종 오차 = 두 오차의 합
        total_loss = loss_confidence + loss_polarity

        return (total_loss, (confidence_logits, polarity_logits)) if return_outputs else total_loss

print("커스텀 Trainer 클래스 정의 완료!")

커스텀 Trainer 클래스 정의 완료!


In [ ]:
from google.colab import drive
import os
import torch

# 1. 구글 드라이브 마운트 (연결)
drive.mount('/content/drive')

# 2. 내 구글 드라이브 안에 저장할 폴더 경로 지정
# '/content/drive/MyDrive/' 까지가 내 드라이브의 최상단(루트) 경로입니다.
drive_save_path = '/content/drive/MyDrive/my_kobert_absa'

Mounted at /content/drive


In [ ]:
# 학습 환경 설정
training_args = TrainingArguments(
    output_dir='./results',          # 모델 가중치가 저장될 폴더
    num_train_epochs=3,              # 전체 데이터 학습 횟수
    per_device_train_batch_size=16,  # 한 번에 학습할 데이터 개수 (코랩 GPU 메모리에 맞춰 조절)
    logging_steps=10,                # 10스텝마다 로그 출력
    save_strategy="epoch",           # 에포크마다 모델 저장
    learning_rate=3e-5,              # BERT 미세조정에 적합한 학습률

    # 💡 핵심 추가 옵션: 모델 forward에 없는 이름의 데이터라도 절대 지우지 마!
    remove_unused_columns=False,
)

# 우리가 개조한 ABSATrainer에 조립
trainer = ABSATrainer(
    model=model,                     # 이전 스텝에서 만든 ABSAKoBERT 객체
    args=training_args,
    train_dataset=train_dataset,     # 1단계에서 만든 데이터셋
)

# 학습 시작!
trainer.train()

# 학습 Report (성능 평가) 출력하기
import torch
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

# 1. 평가 모드 전환 (Dropout 등을 끔)
model.eval()

# 2. 테스트 데이터를 배치 단위로 불러오기 위한 DataLoader 설정
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

true_conf, pred_conf = [], []
true_pol, pred_pol = [], []

# 3. 모델에 테스트 데이터를 넣고 결과 뽑기
with torch.no_grad(): # 평가할 때는 기울기(Gradient) 계산을 하지 않아 메모리를 절약합니다.
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # 모델 예측
        conf_logits, pol_logits = model(input_ids, attention_mask)

        # 가장 높은 확률의 인덱스를 정답으로 채택
        conf_preds = torch.argmax(conf_logits, dim=1).cpu().numpy()
        pol_preds = torch.argmax(pol_logits, dim=1).cpu().numpy()

        # 실제 정답과 예측값 저장
        pred_conf.extend(conf_preds)
        true_conf.extend(batch['confidence_labels'].numpy())

        # 속성이 존재(1)하는 경우에 대해서만 감성 정답/예측값을 수집합니다.
        for i, conf_val in enumerate(batch['confidence_labels'].numpy()):
            if conf_val == 1:
                pred_pol.append(pol_preds[i])
                true_pol.append(batch['polarity_labels'][i].numpy())

# 4. 리포트 출력
print("=== [1] Aspect Confidence (속성 존재 여부) Report ===")
print(classification_report(true_conf, pred_conf, target_names=["없음(0)", "있음(1)"]))

print("\n=== [2] Sentiment Polarity (감성 극성) Report ===")
# 0(부정), 1(중립), 2(긍정)으로 변환했던 라벨을 원래의 -1, 0, 1 의미로 매핑하여 출력
print(classification_report(true_pol, pred_pol, target_names=["부정(-1)", "중립(0)", "긍정(1)"]))

# 해당 경로에 폴더가 없으면 새로 만듭니다.
os.makedirs(drive_save_path, exist_ok=True)

# 3. 모델 가중치 및 토크나이저 저장
torch.save(model.state_dict(), f"{drive_save_path}/model_weights.pth")
tokenizer.save_pretrained(drive_save_path)

print(f"구글 드라이브 경로({drive_save_path})에 모델과 토크나이저가 안전하게 저장되었습니다!")

Step,Training Loss
10,1.099893
20,1.057315
30,1.055705
40,1.130788
50,0.966560
60,0.966164
70,1.060741
80,1.017364
90,1.002705
100,0.991860


=== [1] Aspect Confidence (속성 존재 여부) Report ===
              precision    recall  f1-score   support

       없음(0)       0.94      0.80      0.86     78121
       있음(1)       0.50      0.78      0.61     19531

    accuracy                           0.80     97652
   macro avg       0.72      0.79      0.74     97652
weighted avg       0.85      0.80      0.81     97652


=== [2] Sentiment Polarity (감성 극성) Report ===
              precision    recall  f1-score   support

      부정(-1)       0.79      0.74      0.76      4336
       중립(0)       0.56      0.04      0.07       615
       긍정(1)       0.91      0.96      0.93     14580

    accuracy                           0.88     19531
   macro avg       0.75      0.58      0.59     19531
weighted avg       0.87      0.88      0.87     19531

구글 드라이브 경로(/content/drive/MyDrive/my_kobert_absa)에 모델과 토크나이저가 안전하게 저장되었습니다!
